# Replication on Colab (GPU)

Trains the encoders and attackers on a free GPU.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU → Save*.
2. Put `processed_data.zip` (made in Codespaces, see the README) in your Google Drive, in a folder called `comp8240`.

Then run the cells top to bottom (*Runtime → Run all* works too).

### 1. Connect Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

### 2. Get the code from GitHub

In [ ]:
REPO = 'https://github.com/aayushi-patel13/demog-leakage.git'
!rm -rf /content/demog-leakage && git clone -q $REPO /content/demog-leakage
%cd /content/demog-leakage
!pip -q install -r requirements.txt
!git log --oneline -1

### 3. Load the prepared data from Drive

Results are written to `MyDrive/comp8240/results`, so they survive if Colab disconnects.

In [ ]:
import os, zipfile
D = '/content/drive/MyDrive/comp8240'
z = f'{D}/processed_data.zip'
assert os.path.exists(z), 'processed_data.zip not found in MyDrive/comp8240 - upload it first'
zipfile.ZipFile(z).extractall('data/processed')
os.makedirs(f'{D}/results', exist_ok=True)
!rm -rf results && ln -s $D/results results
!ls -la data/processed/*/ | head -30

### 4. The four replication experiments (about 45-60 minutes)

E1 sentiment-only encoder, E2 race-only encoder, E3 attacker on the E1 encoder (leakage without a defence), E4 adversarial training (lambda = 1) followed by the attacker. Progress is printed after every epoch.

In [ ]:
!python src/experiments.py run-all --data data/processed/sent_race --epochs 20 --tag twitteraae

In [ ]:
print(open('results/twitteraae/results.md').read())

### 5. Optional: the same experiments on the original-as-written preprocessing

Shows whether the quirks of the original data pipeline change the conclusions.

In [ ]:
!python src/experiments.py run-all --data data/processed/sent_race_faithful --epochs 20 --tag twitteraae_faithful
print(open('results/twitteraae_faithful/results.md').read())

### 6. Bring the results back

The folder `MyDrive/comp8240/results/` now holds `results.md`, `results.json` and `log.txt` per run. Download it and add it to the repository's `results/` folder (drag into Codespaces, then commit).